# Camada Bronze — Diagnóstico Consolidado e Integridade Referencial

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026

Consolida os perfis das 8 tabelas Bronze (`data/bronze/*_profile.json`), valida a cobertura referencial cruzada de `id_aluno` frente à tabela `alunos.parquet` e exporta `resumo_bronze.csv` e `relatorio_qualidade_bronze.json`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.bronze_utils import get_project_paths

_, _, BRONZE_DIR = get_project_paths()

alunos_ids = set(
    pd.read_parquet(BRONZE_DIR / "alunos.parquet", columns=["id_aluno"])["id_aluno"]
    .str.strip()
    .str.lstrip("0")
    .replace("", "0")
)

tables = ["alunos", "matriculas", "notas", "acessos", "redes", "financeiro", "deslocamento", "frequencia"]
consolidated = {}
rows = []

for name in tables:
    profile_path = BRONZE_DIR / f"{name}_profile.json"
    with open(profile_path, "r", encoding="utf-8") as f:
        prof = json.load(f)

    df_ids = pd.read_parquet(BRONZE_DIR / f"{name}.parquet", columns=["id_aluno"])["id_aluno"].str.strip()
    non_empty = set(df_ids[df_ids != ""].str.lstrip("0").replace("", "0"))
    match_pct = round(100.0 * len(non_empty & alunos_ids) / len(non_empty), 2) if non_empty else 0.0
    prof["id_aluno_match_alunos_pct"] = match_pct
    consolidated[name] = prof

    rows.append({
        "tabela": name,
        "linhas": prof["metadata"]["rows"],
        "colunas": len(prof["metadata"]["columns"]),
        "id_aluno_unicos": prof["unique_id_aluno"],
        "match_alunos_pct": match_pct,
        "duplicatas_exatas": prof["exact_duplicates"],
        "total_campos_vazios": prof["total_empty_fields"],
    })

resumo_df = pd.DataFrame(rows)
resumo_df.to_csv(BRONZE_DIR / "resumo_bronze.csv", index=False, encoding="utf-8-sig")
with open(BRONZE_DIR / "relatorio_qualidade_bronze.json", "w", encoding="utf-8") as f:
    json.dump(consolidated, f, ensure_ascii=False, indent=2)

print(resumo_df.to_string(index=False))

      tabela  linhas  colunas  id_aluno_unicos  match_alunos_pct  duplicatas_exatas  total_campos_vazios
      alunos 1000000        7          1000000             100.0                  0                17041
  matriculas  700000        6           376925             100.0                  0                    0
       notas  600000        4           349170             100.0                  0                    0
     acessos  400000        4           275613             100.0                  0                    0
       redes  300000        4           225241             100.0                  0                    0
  financeiro 1000000        9            14001             100.0                  0              1503659
deslocamento    1000        9              996             100.0                  0                    0
  frequencia      20        8                9             100.0                  0                    0
